# BKMeeting AI Hub Option 1 Phase 4 Gate

This notebook is the dedicated `Phase 4` benchmark and recommendation notebook for `Option 1`.

Use it after Phase 2 and Phase 3 records already exist.
It should never compile models.


## Environment Notes

Before running this notebook, make sure the current environment can already execute the local `python-model-test` helpers and that the target records for the chosen `RUN_LABEL` already exist.

This notebook only reuses compiled targets and reruns gate logic.


In [ ]:
import sys
from pathlib import Path

import qai_hub as hub

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_hybrid_pipeline import (
    run_vpcd_hybrid_evaluation,
    run_zipformer_hybrid_evaluation,
)
from tools.aihub_option1_phase4_gate import (
    build_phase4_gate_config,
    run_phase4_gate,
    summarize_phase4_gate_reports,
)
from tools.aihub_option1_pilots import (
    build_option1_runtime_config,
    resolve_qai_hub_api_token,
)

API_TOKEN = resolve_qai_hub_api_token(repo_root=Path.cwd())

if not API_TOKEN:
    print("Set QAI_HUB_API_TOKEN in .env or your shell environment before running Qualcomm AI Hub jobs.")
else:
    import os
    os.environ["QAI_HUB_API_TOKEN"] = API_TOKEN
    print("Loaded QAI_HUB_API_TOKEN from .env or shell environment.")
    print("AI Hub device count:", len(hub.get_devices()))


In [ ]:
DEVICE_NAME = "Samsung Galaxy S24 (Family)"
QAIRT_VERSION = None
ZIPFORMER_RUN_LABEL = "20260513-1am"
VPCD_RUN_LABEL = "20260519-aimet-local-quality-parity-notebook"

ENABLE_ZIPFORMER = True
ENABLE_VPCD = True

ZIPFORMER_TARGET_MODEL_ID = None
VPCD_TARGET_MODEL_ID = None
VPCD_PHASE2_COMPILE_PILOT_NAME = "vpcd_option1_local_aimet"

RUNTIME_CONFIG = build_option1_runtime_config(
    device_name=DEVICE_NAME,
    qairt_version=QAIRT_VERSION,
    repo_root=Path.cwd(),
)

print("device:", RUNTIME_CONFIG.device_name)
print("qairt_version:", RUNTIME_CONFIG.qairt_version)
print("record_root:", RUNTIME_CONFIG.record_root)
print("zipformer run label:", ZIPFORMER_RUN_LABEL)
print("vpcd run label:", VPCD_RUN_LABEL)
print("enable zipformer:", ENABLE_ZIPFORMER)
print("enable vpcd:", ENABLE_VPCD)
print("zipformer target model id override:", ZIPFORMER_TARGET_MODEL_ID)
print("vpcd target model id override:", VPCD_TARGET_MODEL_ID)
print("vpcd phase2 compile pilot override:", VPCD_PHASE2_COMPILE_PILOT_NAME)


## Phase 4 Config


In [ ]:
PHASE4_ZIPFORMER_ITERATIONS = 3
PHASE4_VPCD_ITERATIONS = 2
PHASE4_ZIPFORMER_MAX_SAMPLES = 2
PHASE4_VPCD_MAX_SAMPLES = 2
VPCD_PHASE4_MAX_DECODE_STEPS = 5

PHASE4_MINOR_TEXT_DRIFT_THRESHOLD = 0.12
PHASE4_CATASTROPHIC_SHORT_GENERATED_ID_COUNT = 3
PHASE4_CATASTROPHIC_EXPECTED_TEXT_MIN_CHARS = 12
PHASE4_ZIPFORMER_GO_MAX_CLOUD_SECONDS = 240.0
PHASE4_ZIPFORMER_WARN_MAX_CLOUD_SECONDS = 360.0
PHASE4_VPCD_GO_MAX_CLOUD_SECONDS = 480.0
PHASE4_VPCD_WARN_MAX_CLOUD_SECONDS = 900.0

PHASE4_CONFIG = build_phase4_gate_config(
    minor_text_drift_threshold=PHASE4_MINOR_TEXT_DRIFT_THRESHOLD,
    catastrophic_short_generated_id_count=PHASE4_CATASTROPHIC_SHORT_GENERATED_ID_COUNT,
    catastrophic_expected_text_min_chars=PHASE4_CATASTROPHIC_EXPECTED_TEXT_MIN_CHARS,
    zipformer_go_max_average_cloud_inference_seconds=PHASE4_ZIPFORMER_GO_MAX_CLOUD_SECONDS,
    zipformer_warn_max_average_cloud_inference_seconds=PHASE4_ZIPFORMER_WARN_MAX_CLOUD_SECONDS,
    vpcd_go_max_average_cloud_inference_seconds=PHASE4_VPCD_GO_MAX_CLOUD_SECONDS,
    vpcd_warn_max_average_cloud_inference_seconds=PHASE4_VPCD_WARN_MAX_CLOUD_SECONDS,
)

print("phase4 zipformer iterations:", PHASE4_ZIPFORMER_ITERATIONS)
print("phase4 vpcd iterations:", PHASE4_VPCD_ITERATIONS)
print("phase4 zipformer max samples:", PHASE4_ZIPFORMER_MAX_SAMPLES)
print("phase4 vpcd max samples:", PHASE4_VPCD_MAX_SAMPLES)
print("phase4 vpcd max decode steps:", VPCD_PHASE4_MAX_DECODE_STEPS)
print("phase4 gate config:", PHASE4_CONFIG)


### Zipformer Phase 4 Benchmark And Gate


In [ ]:
zipformer_phase4_report = None
if ENABLE_ZIPFORMER:
    zipformer_compile_record_path = RUNTIME_CONFIG.pilot_record_dir("zipformer_encoder_option1") / f"compile-run-{ZIPFORMER_RUN_LABEL}.json"
    zipformer_hybrid_record_path = RUNTIME_CONFIG.pilot_record_dir("zipformer_hybrid_option1") / f"hybrid-run-{ZIPFORMER_RUN_LABEL}.json"
    print("zipformer compile record input:", zipformer_compile_record_path)
    print("zipformer hybrid record input:", zipformer_hybrid_record_path)
    zipformer_phase4_report = run_phase4_gate(
        pilot_name="zipformer",
        runtime_config=RUNTIME_CONFIG,
        hybrid_runner=lambda **kwargs: run_zipformer_hybrid_evaluation(
            runtime_config=RUNTIME_CONFIG,
            run_label=kwargs.get("run_label"),
            explicit_target_model_id=kwargs.get("explicit_target_model_id"),
            max_samples=kwargs.get("max_samples", PHASE4_ZIPFORMER_MAX_SAMPLES),
        ),
        iterations=PHASE4_ZIPFORMER_ITERATIONS,
        max_samples=PHASE4_ZIPFORMER_MAX_SAMPLES,
        run_label=ZIPFORMER_RUN_LABEL,
        explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
        config=PHASE4_CONFIG,
    )

    print("zipformer phase4 record:", zipformer_phase4_report["record_path"])
    print("zipformer phase4 warmup:", zipformer_phase4_report["benchmark_summary"]["warmup"])
    print("zipformer phase4 steady state:", zipformer_phase4_report["benchmark_summary"]["steady_state"])
    print("zipformer phase4 severity counts:", zipformer_phase4_report["correctness_summary"]["severity_counts"])
    print("zipformer phase4 recommendation:", zipformer_phase4_report["recommendation"])
    print("zipformer phase4 per-sample severity:")
    for row in zipformer_phase4_report["correctness_summary"]["sample_results"]:
        print(row)
else:
    print("Skipping Zipformer Phase 4 because ENABLE_ZIPFORMER is False.")


### VPCD Phase 4 Benchmark And Gate


In [ ]:
vpcd_phase4_report = None
if ENABLE_VPCD:
    vpcd_prepared_record_path = RUNTIME_CONFIG.pilot_record_dir(VPCD_PHASE2_COMPILE_PILOT_NAME) / f"prepared-artifact-{VPCD_RUN_LABEL}.json"
    vpcd_compile_record_path = RUNTIME_CONFIG.pilot_record_dir(VPCD_PHASE2_COMPILE_PILOT_NAME) / f"compile-run-{VPCD_RUN_LABEL}.json"
    vpcd_live_record_path = RUNTIME_CONFIG.pilot_record_dir(VPCD_PHASE2_COMPILE_PILOT_NAME) / f"live-run-{VPCD_RUN_LABEL}.json"
    vpcd_hybrid_record_path = RUNTIME_CONFIG.pilot_record_dir("vpcd_hybrid_option1") / f"hybrid-run-{VPCD_RUN_LABEL}.json"
    print("vpcd prepared record input:", vpcd_prepared_record_path)
    print("vpcd compile record input:", vpcd_compile_record_path)
    print("vpcd live record input:", vpcd_live_record_path)
    print("vpcd hybrid record input:", vpcd_hybrid_record_path)
    vpcd_phase4_report = run_phase4_gate(
        pilot_name="vpcd",
        runtime_config=RUNTIME_CONFIG,
        hybrid_runner=lambda **kwargs: run_vpcd_hybrid_evaluation(
            runtime_config=RUNTIME_CONFIG,
            run_label=kwargs.get("run_label"),
            explicit_target_model_id=kwargs.get("explicit_target_model_id"),
            compile_pilot_name=VPCD_PHASE2_COMPILE_PILOT_NAME,
            max_samples=kwargs.get("max_samples", PHASE4_VPCD_MAX_SAMPLES),
            max_decode_steps=VPCD_PHASE4_MAX_DECODE_STEPS,
        ),
        iterations=PHASE4_VPCD_ITERATIONS,
        max_samples=PHASE4_VPCD_MAX_SAMPLES,
        run_label=VPCD_RUN_LABEL,
        explicit_target_model_id=VPCD_TARGET_MODEL_ID,
        phase2_compile_pilot_name_override=VPCD_PHASE2_COMPILE_PILOT_NAME,
        config=PHASE4_CONFIG,
    )

    print("vpcd phase4 record:", vpcd_phase4_report["record_path"])
    print("vpcd phase4 warmup:", vpcd_phase4_report["benchmark_summary"]["warmup"])
    print("vpcd phase4 steady state:", vpcd_phase4_report["benchmark_summary"]["steady_state"])
    print("vpcd phase4 severity counts:", vpcd_phase4_report["correctness_summary"]["severity_counts"])
    print("vpcd phase4 recommendation:", vpcd_phase4_report["recommendation"])
    print("vpcd phase4 per-sample severity:")
    for row in vpcd_phase4_report["correctness_summary"]["sample_results"]:
        print(row)
else:
    print("Skipping VPCD Phase 4 because ENABLE_VPCD is False.")


## Phase 4 Recommendation Summary


In [ ]:
phase4_reports = [
    report
    for report in [globals().get("zipformer_phase4_report"), globals().get("vpcd_phase4_report")]
    if report is not None
]
phase4_summary = summarize_phase4_gate_reports(phase4_reports) if phase4_reports else None

print("phase4 overall summary:", phase4_summary)
print("zipformer run label:", ZIPFORMER_RUN_LABEL)
print("vpcd run label:", VPCD_RUN_LABEL)
if globals().get("zipformer_phase4_report") is not None:
    print("zipformer recommendation:", zipformer_phase4_report["recommendation"])
else:
    print("zipformer recommendation: skipped")
if globals().get("vpcd_phase4_report") is not None:
    print("vpcd recommendation:", vpcd_phase4_report["recommendation"])
else:
    print("vpcd recommendation: skipped")
print("phase4 reminder: Phase 5 still packages every pilot, but Android-facing work should only consume justified deployment candidates.")
